# Project imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

from datetime import datetime
import time
from fontTools.merge.util import current_time

from Talpha import AdaptiveLayer

# initialize flappy game

In [ ]:
import sys
import os
import pygame
import warnings
warnings.filterwarnings("ignore", message="pkg_resources is deprecated")

os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

sys.path.insert(0, '../itml-project2')
# noinspection PyUnresolvedReferences
from ple.games.flappybird import FlappyBird
# noinspection PyUnresolvedReferences
from ple import PLE

In [ ]:
class PPO_Flappy(nn.Module):

    def __init__(self, hidden_layers, layer_configs=None, **adaptive_kwargs):
        super().__init__()
        sizes = [8] + hidden_layers
        self.hidden = nn.ModuleList([
            nn.Linear(sizes[i], sizes[i + 1]) for i in range(len(sizes) - 1)
        ])
        if layer_configs is not None:
            self.adaptive = nn.ModuleList([
                AdaptiveLayer(size, **cfg) for size, cfg in zip(hidden_layers, layer_configs)
            ])
        else:
            self.adaptive = nn.ModuleList([
                AdaptiveLayer(size, **adaptive_kwargs) for size in hidden_layers
            ])
        self.actor = nn.Linear(hidden_layers[-1], 2)
        self.critic = nn.Linear(hidden_layers[-1], 1)

    def forward(self, z):
        for linear, adaptive in zip(self.hidden, self.adaptive):
            z = adaptive(linear(z))
        action_prob = F.softmax(self.actor(z), dim=-1)
        state_value = self.critic(z)
        return action_prob, state_value

# Hyper parameters

In [ ]:
lr = 0.0003
epochs = 200
K_epochs = 4

epsilon = 0.2
gamma = 0.99
lam = 0.95
target_steps = 8192
minibatch_size = 512
max_grad_norm = 0.5
c0 = 1.0
c1 = 0.5
c2 = 0.01


# Model initialization

In [ ]:
model = PPO_Flappy([1024, 128, 16, 256, 256])

load = False

if load:
    try:
        model.load_state_dict(torch.load('Weights/Old/AT_L2.pt'))
    except FileNotFoundError:
        print("No weights found in: flappy weights/")
else:
    print("Training from scratch")

optimizer = torch.optim.Adam(model.parameters(),lr = lr)

# Helper Functions

In [ ]:
def normalize_game_state(state):
    means = torch.tensor([150.0, 0.0, 76.0, 108.0, 208.0, 226.0, 108.0, 208.0])
    stds = torch.tensor([44.0, 5.0, 44.0, 48.0, 48.0, 44.0, 48.0, 48.0])
    return (state - means) / stds


def compute_advantage(rewards, values, gamma, lam):
    t_max = len(rewards)
    advantages = torch.zeros(t_max, dtype=torch.float32)
    last_gae = 0.0
    for t in reversed(range(t_max)):
        next_value = values[t + 1] if t < t_max - 1 else 0.0
        delta = rewards[t] + gamma * next_value - values[t]
        last_gae = delta + gamma * lam * last_gae
        advantages[t] = last_gae
    values_targ = advantages + values
    return advantages.detach(), values_targ.detach()


# training Loop

In [ ]:
import time

print_freq = 100

start_time = time.time()
game = FlappyBird()
p = PLE(game, fps=30, display_screen=False, force_fps=True,
        reward_values={'positive': 1.0, 'tick': 0.0, 'loss': -5.0})
p.init()

# Block QUIT events so the headless dummy window can't trigger sys.exit()
pygame.event.set_blocked(pygame.QUIT)

epoch_pipes = []
all_L_clip = []
all_L_vf = []
all_L_entropy = []

for epoch in range(epochs):

    # Anneal learning rate
    cur_lr = lr * (1.0 - epoch / epochs)
    for param_group in optimizer.param_groups:
        param_group['lr'] = cur_lr

    # --- Collect batch of episodes until target_steps reached ---
    batch_states = []
    batch_actions = []
    batch_logprobs = []
    batch_values = []
    batch_rewards = []
    batch_ep_lengths = []
    total_steps = 0
    episode_pipes = []

    while total_steps < target_steps:
        p.reset_game()
        ep_states, ep_actions, ep_logprobs, ep_values, ep_rewards = [], [], [], [], []
        nr_pipes = 0

        while not p.game_over():
            state = normalize_game_state(torch.tensor(list(p.getGameState().values())))
            with torch.no_grad():
                action_prob, critic_val = model(state)

            dist = torch.distributions.Categorical(action_prob.squeeze())
            action = dist.sample()

            reward = p.act(p.getActionSet()[action.item()])
            if reward > 0:
                nr_pipes += 1

            ep_states.append(state)
            ep_actions.append(action)
            ep_logprobs.append(dist.log_prob(action))
            ep_values.append(critic_val.squeeze())
            ep_rewards.append(reward)

        episode_pipes.append(nr_pipes)
        ep_len = len(ep_states)
        total_steps += ep_len

        batch_states.append(torch.stack(ep_states))
        batch_actions.append(torch.stack(ep_actions))
        batch_logprobs.append(torch.stack(ep_logprobs))
        batch_values.append(torch.stack(ep_values))
        batch_rewards.append(torch.tensor(ep_rewards, dtype=torch.float32))
        batch_ep_lengths.append(ep_len)

    avg_pipes = sum(episode_pipes) / len(episode_pipes)
    epoch_pipes.append(avg_pipes)

    # --- Compute GAE advantages per episode, then concatenate ---
    all_adv = []
    all_vtarg = []
    for i in range(len(batch_rewards)):
        adv, vtarg = compute_advantage(batch_rewards[i], batch_values[i], gamma, lam)
        all_adv.append(adv)
        all_vtarg.append(vtarg)

    states = torch.cat(batch_states)
    actions = torch.cat(batch_actions)
    logp_old = torch.cat(batch_logprobs)
    advantages = torch.cat(all_adv)
    values_targ = torch.cat(all_vtarg)

    # Normalize advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    # --- Minibatch PPO training ---
    for k in range(K_epochs):
        indices = torch.randperm(states.shape[0])
        for start in range(0, states.shape[0] - minibatch_size + 1, minibatch_size):
            mb = indices[start:start + minibatch_size]

            action_probs, new_values = model(states[mb])
            new_dist = torch.distributions.Categorical(action_probs)
            new_logp = new_dist.log_prob(actions[mb])

            ratio = torch.exp(new_logp - logp_old[mb])
            surr1 = ratio * advantages[mb]
            surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantages[mb]

            L_clip = torch.min(surr1, surr2).mean()
            L_vf = F.mse_loss(new_values.squeeze(), values_targ[mb])
            entropy = new_dist.entropy().mean()

            loss = -(c0 * L_clip - c1 * L_vf + c2 * entropy)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        all_L_clip.append(L_clip.item()*c0)
        all_L_vf.append(L_vf.item()*c1)
        all_L_entropy.append(entropy.item()*c2)

    # --- Print stats ---
    if (epoch + 1) % print_freq == 0:
        elapsed = time.time() - start_time
        n = min(print_freq, len(epoch_pipes))
        recent_pipes = epoch_pipes[-n:]
        n_loss = min(K_epochs * print_freq, len(all_L_clip))

        print(f"Epoch {epoch+1:5d} | "
              f"Avg Pipes: {sum(recent_pipes)/len(recent_pipes):7.2f} | "
              f"L_clip: {sum(all_L_clip[-n_loss:])/len(all_L_clip[-n_loss:]):.4f} | "
              f"L_vf: {sum(all_L_vf[-n_loss:])/len(all_L_vf[-n_loss:]):.4f} | "
              f"L_ent: {sum(all_L_entropy[-n_loss:])/len(all_L_entropy[-n_loss:]):.4f} | "
              f"lr: {cur_lr:.6f} | "
              f"Time: {elapsed:.1f}s")

# Plotting Stats

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.scatter(range(len(epoch_pipes)), epoch_pipes, s=1, alpha=0.3, label='Avg pipes per epoch')

# Rolling average
window = 50
if len(epoch_pipes) >= window:
    rolling_avg = [sum(epoch_pipes[max(0,i-window):i+1]) / len(epoch_pipes[max(0,i-window):i+1])
                   for i in range(len(epoch_pipes))]
    ax.plot(rolling_avg, color='red', linewidth=1.5, label=f'Rolling avg (window={window})')

ax.set_xlabel('Epoch')
ax.set_ylabel('Avg Pipes')
ax.set_title(f'Training Progress — Best Avg: {max(epoch_pipes):.1f} pipes')
ax.legend()
plt.tight_layout()
plt.show()


# Saving Model

In [ ]:
torch.save(model.state_dict(), "Weights/Old/AT_L2.pt")